In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

df = pd.read_csv("../data/Housing_dataset.csv")

print("Shape: ", df.shape)
df.head()

In [ ]:
# Data types
print(df.dtypes)

# Missing values
print(f"\nMissing values\n: {df.isnull().sum()}")

# Duplicates
print(f"Duplicates: {df.duplicated().sum()}")

# Unique values per column (categorical sanity)
for col in df.columns:
    print(f"{col:20s} --> {df[col].nunique():4d} unique")



In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(16, 4))

# Raw distribution
sns.histplot(df["price"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Price Distribution (raw)")
axes[0].axvline(df["price"].mean(), color="red", linestyle="--", label="mean")
axes[0].axvline(df["price"].median(), color="green", linestyle="--", label="median")
axes[0].legend()

# log transformed
sns.histplot(np.log1p(df["price"]), kde=True, ax=axes[1], color="darkorange")
axes[0].set_title("log1p price Distribution")

# Boxplot -- outlier view
sns.boxplot(x=df["price"], ax=axes[2], color="lightgreen")
axes[2].set_title("Price Boxplot")

plt.tight_layout()
os.makedirs("../reports/figures", exist_ok=True)
plt.savefig("../reports/figures/01-price-distribution.png", dpi=120)
plt.show()

print(df["price"].describe())
print(df["price"].skew().round(2))
print(df["price"].kurt().round(2))

In [ ]:
num_cols = ["area", "bedrooms", "bathrooms", "stories", "parking"]

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.scatterplot(data=df, x=col, y="price", ax=axes[i], alpha=0.6)
    axes[i].set_title(f"{col} vs Price")
    axes[i].set_ylabel("price")
axes[-1].axis("off")
plt.tight_layout()
plt.savefig("../reports/figures/02-numeric_vs_price.png", dpi=120)
plt.show()

print(df[num_cols + ["price"]].corr()["price"].sort_values(ascending=False))

In [ ]:
cat_cols = ["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea", "furnishingstatus"]

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    sns.boxplot(data=df, x=col, y="price", ax=axes[i])
    axes[i].set_title(f"{col}")

axes[-1].axis("off")
axes[-2].axis("off")
plt.tight_layout()
plt.savefig("../reports/figures/03-categorical_vs_price.png", dpi=120)
plt.show()

for col in cat_cols:
    print(f"\n---- {col} ----")
    print(df.groupby(col)["price"].agg(["median", "mean", "count"]).round(0))

In [ ]:
# Rare categories check
for col in ["hotwaterheating", "guestroom", "basement"]:
    print(df[col].value_counts(normalize=True).round(3))

In [ ]:
# Binary mapping
binary_cols = ["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea"]
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().str.strip()
        df[col] = df[col].map({"yes": 1, "no": 0}).fillna(0).astype(int)

# One-Hot Encoding (multi-categories)
if "furnishingstatus" in df.columns:
    df = pd.get_dummies(df, columns=['furnishingstatus'], dtype=int)


# Correlation Matrix
corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heapmap of Processed data")
plt.tight_layout()
plt.savefig("../reports/figures/04-Correlation_heatmap.png", dpi=300)
plt.show()
plt.close()

In [ ]:
# Naive Model

from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

X = df.drop(columns="price")
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def report(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{name:15s} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | R2: {r2:.3f}")

dummy = DummyRegressor(strategy='median').fit(X_train, y_train)
report("T0 Dummpy:", y_test, dummy.predict(X_test))


In [ ]:
# Simple Model

from sklearn.linear_model import LinearRegression

area_model = LinearRegression().fit(X_train[["area"]], y_train)
report("T1 Dummpy:", y_test, area_model.predict(X_test[["area"]]))

In [ ]:
# Standard Model

X_train_enc = pd.get_dummies(X_train, drop_first=True)
X_test_enc = pd.get_dummies(X_test, drop_first=True)

X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

t2 = LinearRegression().fit(X_train_enc, y_train)
report("T2 Dummpy:", y_test, t2.predict(X_test_enc))

In [ ]:
# Coefficient interpretation

coefs = pd.Series(t2.coef_, index=X_train_enc.columns).sort_values()
print(coefs)

fig, ax = plt.subplots(figsize=(8, 6))
coefs.plot.barh(ax=ax)
ax.set_xlabel("Rs per unit change")
ax.set_title("Linear Model Coefficients (naive)")
plt.tight_layout()
plt.savefig("../reports/figures/05-Linear_coefs.png", dpi=150)
plt.show()